# START

In [2]:
import sys
!{sys.executable} -m pip install mcp


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## MCP_stata.py

In [4]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

In [6]:
def create_server_params(work_dir: str | None = None):
    """封装stata-code MCP服务的启动参数"""
    env = None
    if work_dir:
        env = {"STATA_MCP_CWD": work_dir}

    return StdioServerParameters(
        command="stata-code-mcp",
        args=[],
        env=env
    )


async def load_tools(work_dir: str | None = None):
    """启动MCP服务端并返回LangChain兼容的工具列表"""
    server_params = create_server_params(work_dir)

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            return tools


async def main():
    """测试入口：打印所有加载成功的工具名称与描述"""
    print("Connecting to stata-code MCP service...")
    try:
        tools = await load_tools()
        print(f"\nLoaded {len(tools)} tools:")
        for t in tools:
            print(f"  - {t.name}: {t.description[:80]}...")
    except Exception as e:
        print(f"Failed: {e}")


if __name__ == "__main__":
    asyncio.run(main())

Connecting to stata-code MCP service...
Failed: fileno
